# French Motor end-to-end

From raw policies and claims to a rate filing in five steps. The dataset is the French Motor TPL (Charpentier 2014, CASdatasets, CC-BY-NC). When the data isn't cached locally, the synthetic generator stands in with the same schema.

In [ ]:
from pyratemaking import RatePlan
from pyratemaking.datasets import french_motor, synthetic

try:
    policies, claims = french_motor.load()
except OSError:
    policies, claims = synthetic.generate(n_policies=20_000, seed=42)

policies.head()

## 1. Overall rate level indication

In [ ]:
plan = RatePlan(policies=policies, claims=claims)
indication = plan.indicate(method="loss_ratio")
indication.summary()

## 2. Classification GLM

In [ ]:
plan.classify(
    rating_vars=["region", "veh_brand", "veh_gas"],
    family="tweedie",
    backend="glum",
    power=1.5,
)
plan.classification.relativities_frame().head()

## 3. Diagnostics

In [ ]:
plan.diagnostics.lift(n_bins=10).figure()
print(f"Gini: {plan.diagnostics.gini():.4f}")

## 4. Implementation

In [ ]:
plan.implement(cap=1.15, floor=0.85)
plan.implementation.dispersion_summary()

## 5. Filing

In [ ]:
plan.report.filing("filing.html")
plan.report.excel("filing.xlsx")